In [1]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python --version
!nproc
!ls /kaggle/input/

name, memory.total [MiB]
Tesla T4, 15360 MiB
Tesla T4, 15360 MiB
Python 3.12.13
4
datasets


In [2]:
!ls -R /kaggle/input/datasets/ | head -20

/kaggle/input/datasets/:
bornamuzina

/kaggle/input/datasets/bornamuzina:
dataset-negatives-included

/kaggle/input/datasets/bornamuzina/dataset-negatives-included:
splits.json
V_AIRPLANE_001_meta.json
V_AIRPLANE_001_tensors.npz
V_AIRPLANE_002_meta.json
V_AIRPLANE_002_tensors.npz
V_AIRPLANE_003_meta.json
V_AIRPLANE_003_tensors.npz
V_AIRPLANE_004_meta.json
V_AIRPLANE_004_tensors.npz
V_AIRPLANE_005_meta.json
V_AIRPLANE_005_tensors.npz
V_AIRPLANE_006_meta.json
V_AIRPLANE_006_tensors.npz
ls: write error: Broken pipe


In [3]:
!add-apt-repository -y ppa:deadsnakes/ppa > /dev/null 2>&1
!apt-get -qq install -y python3.11 python3.11-dev > /dev/null 2>&1
!python3.11 --version

Python 3.11.16


In [4]:
!git clone -q https://github.com/bornamuzina/akida-drone-detection /kaggle/working/repo
!ls /kaggle/working/repo/src/model

augment_check.jpg	 config.py	     evaluate.py    targets.py
augment.py		 convert_weights.py  fail_reasons   train.py
check_akida_stride16.py  dataset.py	     pretrained.py  yolo_stride16.py


In [7]:
import os, glob, json, shutil

ROOT = '/kaggle/working/drone'
TENSORS = f'{ROOT}/Data_new/tensors_rgb/tensors_rgb_256_packed'
DATA = '/kaggle/input/datasets/bornamuzina/dataset-negatives-included'

os.makedirs(f'{ROOT}/Data_new/tensors_rgb', exist_ok=True)
os.makedirs(f'{ROOT}/Data_new/runs', exist_ok=True)

if not os.path.exists(TENSORS):
    os.symlink(DATA, TENSORS)

shutil.copy('/kaggle/working/repo/Data_new/splits.json',
            f'{ROOT}/Data_new/splits.json')

n = len(glob.glob(f'{TENSORS}/*_tensors.npz'))
s = json.load(open(f'{ROOT}/Data_new/splits.json'))
print('clips:', n)
print('neg:', len(s.get('train_neg', [])), len(s.get('validation_neg', [])), len(s.get('test_neg', [])))
assert n == 285 and s.get('train_neg')

clips: 285
neg: 135 17 19


In [8]:
!pip install -q virtualenv
!virtualenv -q -p python3.11 /kaggle/working/akv
!/kaggle/working/akv/bin/pip install -q -r /kaggle/working/repo/requirements/requirements-akida.txt

!MPLBACKEND=Agg /kaggle/working/akv/bin/python -c "import tensorflow as tf, tf_keras, akida; print('tf', tf.__version__, '| akida', akida.__version__, '| gpu', bool(tf.config.list_physical_devices('GPU')))"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 80.2 MB/s eta 0:00:00ta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 470.6/470.6 kB 23.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
2026-09-23 10:52:02.232734: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790160722.255142    1046 cuda_dnn.cc:8579] Unable to register cuDNN f

In [9]:
!git -C /kaggle/working/repo pull -q
!grep -c freeze /kaggle/working/repo/src/model/train.py
!ls /kaggle/working/repo/src/model/augment.py

16
/kaggle/working/repo/src/model/augment.py


In [14]:
import os, shutil, pathlib
dst = pathlib.Path("/kaggle/working/repo/Data_new/tensors_rgb/tensors_rgb_256_packed")
shutil.rmtree(dst)
os.symlink("/kaggle/input/datasets/bornamuzina/dataset-negatives-included", dst)
print(len(list(dst.iterdir())), "datoteka")
print("splits:", pathlib.Path("/kaggle/working/repo/Data_new/splits.json").exists())

571 datoteka
splits: True


In [15]:
!cd /kaggle/working/repo/src/model && MPLBACKEND=Agg /kaggle/working/akv/bin/python -u train.py \
    --epochs 6 \
    --negatives \
    --pretrained \
    --freeze 2 \
    --augment \
    --name yolov2_s16_rgb_256_neg_pre_frz_aug

2026-09-23 10:57:13.470273: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790161033.493907    1110 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790161033.501493    1110 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790161033.521189    1110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790161033.521216    1110 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790161033.521220    1110 computation_placer.cc:177] computation placer alr

In [16]:
!cd /kaggle/working/repo/src/model && MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py \
    --split test --negatives \
    --run yolov2_s16_rgb_256_neg_pre_frz_aug \
    --weights best_ap.weights.h5

!cd /kaggle/working/repo/src/model && MPLBACKEND=Agg /kaggle/working/akv/bin/python -u evaluate.py \
    --split test --negatives \
    --run yolov2_s16_rgb_256_neg_pre_frz_aug \
    --weights best.weights.h5

2026-09-23 12:39:32.932576: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1790167172.958669    1235 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1790167172.966965    1235 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1790167172.990119    1235 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790167172.990183    1235 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1790167172.990194    1235 computation_placer.cc:177] computation placer alr

In [17]:
import shutil, os
run = "/kaggle/working/repo/src/model/runs/yolov2_s16_rgb_256_neg_pre_frz_aug"
shutil.make_archive("/kaggle/working/neg_pre_frz_aug", "zip", run)
print(os.path.getsize("/kaggle/working/neg_pre_frz_aug.zip") / 1e6, "MB")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/repo/src/model/runs/yolov2_s16_rgb_256_neg_pre_frz_aug'

In [18]:
import shutil, os
run = "/kaggle/working/repo/Data_new/runs/yolov2_s16_rgb_256_neg_pre_frz_aug"
shutil.make_archive("/kaggle/working/neg_pre_frz_aug", "zip", run)
print(os.path.getsize("/kaggle/working/neg_pre_frz_aug.zip") / 1e6, "MB")

53.049573 MB
